# **DEEP LEARNING MODELLING: CNN-LSTM WITHOUT GEOGRAPHICAL COORDINATES**

## **1. LIBRARIES**

### **1.1 DEPENDENCIES**

In [ ]:
! pip install adlfs azure-storage-blob
! pip install tensorflow
! pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.9/187.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 12.5 MB/s eta 0:00:00


### **1.2 LIBRARIES**

In [ ]:
# STANDARD LIBRARY
import os
import io
import random

# DATA MANIPULATION AND ANALYSIS
import pandas as pd
import numpy as np

# DATA VISUALISATION
import matplotlib.pyplot as plt
import seaborn as sns

# AZURE CLOUD STORAGE
from adlfs import AzureBlobFileSystem
from azure.storage.blob import BlobServiceClient

# REGULAR EXPRESSIONS
import re

# TIME SERIES
from sklearn.model_selection import TimeSeriesSplit

# STANDARIZATION
from sklearn.preprocessing import StandardScaler

# EVALUATION METRICS
from sklearn.metrics import r2_score

# PROGRESS BAR
from tqdm import tqdm

# TENSORFLOW
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Conv1D, BatchNormalization, Dropout, LSTM, Dense, Input
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.losses import Huber

# SEED
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [ ]:
print(tf.__version__)

2.19.0


In [ ]:
# Correctly set pandas display option for max columns and rows
pd.options.display.max_columns = None
pd.options.display.max_rows = None

## **2. LOADING DATASETS FROM AZURE BLOB STORAGE**

In [ ]:
def load_csv_from_blob(blob_name):
    blob_client = container_client.get_blob_client(blob_name)
    stream = blob_client.download_blob()
    data_bytes = stream.readall()
    data_str = data_bytes.decode("utf-8")
    return pd.read_csv(io.StringIO(data_str))


# Connection configuration
AZURE_STORAGE_ACCOUNT = "researchprojectx24104515"
AZURE_STORAGE_KEY = "bxpexO6i+Hz6n1WiipTn+sTCuLPGMS1BogMERrIrHd16DpQ0GLfQ0R33yrSw4MxsDomq5yNMgw1o+AStlx/MjA=="
connection_string = f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

fs = AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

In [ ]:
# Establish connection to cycle
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "preprocessed.csv"
data = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {data.shape}")

Loaded 'preprocessed.csv' with shape: (53424, 44)


In [ ]:
# Interaction features created with weather features
interactionf = ["WTH_dew_spread", "WTH_prob_cond", "WTH_severe"]

# Using regex to get all temporal and weather prefix patterns
temp_pattern = re.compile(r"^TEMP_")
wth_patterb = re.compile(r"^WTH_")

# Using regex to get all temporal and weather features
temporalf = [col for col in data.columns if temp_pattern.match(col)]
weatherf = [col for col in data.columns if wth_patterb.match(col) and col not in interactionf]

# Combine all features
fcolumns = weatherf + temporalf + interactionf

# Extract cycle and footfall targets
cycle_t = [col for col in data.columns if col.startswith("CYC_")]
footfall_t = [col for col in data.columns if col.startswith("FTF_")]
targets = cycle_t + footfall_t

# Prepare feature matrix
X = data[fcolumns].copy()
y = data[targets].copy()

# Handle missing values
X = X.ffill().bfill()
y = y.fillna(0)

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")
print(f"Feature columns: {len(fcolumns)}")

Features shape: (53424, 24)
Targets shape: (53424, 19)
Feature columns: 24


## **3. CNN / LSTM**

### **3.1 DEFINE CNN-LSTM SEQUENCES AND TIME SERIES SPLIT**

In [ ]:
# CNN-LSTM sequences
def cnnlstm_seqs(data, targets, features, seq_len=24):

    # Scale all features
    f_scaler = MinMaxScaler()
    features_scaled = f_scaler.fit_transform(data[features])

    # Containers for results
    X_data = {}
    y_data = {}
    t_scalers = {}

    # Processing separately each target
    for t in targets:
        # Extract target column and reshape to 2D
        y = data[t].values.reshape(-1, 1)

        # Fitting a scaler
        t_scaler = MinMaxScaler()
        y_scaled = t_scaler.fit_transform(y)
        X, y_seq = [], []

        # Creating sequences of length
        for i in range(seq_len, len(features_scaled)):
            seq = features_scaled[i-seq_len:i]
            X.append(seq)
            y_seq.append(y_scaled[i])

        # Converting data to NumPy arrays to store in dictionaries
        X_data[t] = np.array(X, dtype=np.float32)
        y_data[t] = np.array(y_seq, dtype=np.float32)
        t_scalers[t] = t_scaler

    return X_data, y_data, f_scaler, t_scalers

# Time Series Split
def ts_split(X, y, train_ratio=0.7, val_ratio=0.15):

    # Total number of sequences
    n_samples = len(X)

    # Index boundaeries calculated by ratios
    train_end = int(n_samples * train_ratio)
    val_end = int(n_samples * (train_ratio + val_ratio))

    # Slice sets sequentially as a Time Series
    X_train = X[:train_end]
    y_train = y[:train_end]
    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]
    X_test = X[val_end:]
    y_test = y[val_end:]

    return X_train, X_val, X_test, y_train, y_val, y_test

### **3.2 BUILD CNN-LSTM MODEL**

In [ ]:
def build_cnnlstm(input_shape=(24, 24), conv_filters=64, lstm_units=64, dense_units=64, dropout_rate=0.2):
    # Level 2 regularisation
    reg = 1e-4

    # Batch Normalization for Training stabilisation and Dropout to avoid overfitting

    # Model
    model = Sequential()
    model.add(Input(shape=input_shape))

    # FEATURE EXTRACTION

    # First convolutional layer for feature extraction from short-term sequential data
    model.add(Conv1D(conv_filters, 3, activation="relu", padding="same",
                     kernel_regularizer=regularizers.l2(reg)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # Second convolutional layer for more abstract features and long-term temporal patterns
    model.add(Conv1D(conv_filters//2, 5, activation="relu", padding="same",
                     kernel_regularizer=regularizers.l2(reg)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # LSTM FOR LONG TERM DEPENDENCIES

    # First LSTM layer, returns to a sequence that keeps temporal information
    model.add(LSTM(lstm_units, return_sequences=True, dropout=dropout_rate, recurrent_dropout=dropout_rate,
                   kernel_regularizer=regularizers.l2(reg)))
    model.add(Dropout(dropout_rate))

    # Second LSTM layer, returns to a single output
    model.add(LSTM(lstm_units//2, return_sequences=False, dropout=dropout_rate, recurrent_dropout=dropout_rate,
                   kernel_regularizer=regularizers.l2(reg)))
    model.add(Dropout(dropout_rate))

    # Dense layer for learned features before final prediction
    model.add(Dense(dense_units, activation="relu", kernel_regularizer=regularizers.l2(reg)))
    model.add(Dropout(dropout_rate))

    # Single output neuron with no negative predictions
    model.add(Dense(1, activation="relu"))

    # Model compilation
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=Huber(delta=1.0),
        metrics=["mae"]
    )
    return model

### **3.3 TRAIN MODEL**

In [ ]:
# TRAIN MODEL
def train_cnnlstm(target_name, X, y, callbacks=None):

    # Training starts
    print(f"\nTraining model for: {target_name} ===")

    # Split the sequences into train, test and validation sets
    X_train, X_val, X_test, y_train, y_val, y_test = ts_split(X, y)

    # CNN-LSTM model built with an input shape that matches the sequences
    model = build_cnnlstm(input_shape=X_train.shape[1:])

    # If not callbacks provided, use this as default
    if callbacks is None:
        # Stop if validation loss stops improving for 10 epochs
        early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        # Reduce learning rate if validation loss doe not improve after 5 epochs
        reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
        callbacks = [early_stopping, reduce_lr]

    # Train the model
    history = model.fit(
        X_train, y_train,
        # Maximum training Epochs
        epochs=100,
        # Large batch size for training efficiency
        batch_size=1024,
        # Monitor performance on validation set
        validation_data=(X_val, y_val),
        # Apply early stop or reduce learning rate
        callbacks=callbacks,
        # Shows training process
        verbose=1
    )

    print(f"------Training completed for: {target_name}")
    return model, history, X_test, y_test

### **3.4 EVALUTAION**

In [ ]:
# EVALUATE MODEL
def evaluate_mtarget(model, X_test, y_test, target_scaler, target_name, target_idx, n_targets):
    print(f"\n----- Evaluating model for: {target_name} ({target_idx + 1}/{n_targets})")

    y_pred = model.predict(X_test)

    # Inverse transfomation of y_pred and predictions
    y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1))
    y_pred = target_scaler.inverse_transform(y_pred)

    # Round to whole numbers
    y_pred = np.round(y_pred).astype(int)
    y_true = np.round(y_true).astype(int)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if np.any(mask) else np.nan
    r2 = r2_score(y_true, y_pred)

    print("Evaluation Metrics:")
    print(f" -RMSE : {rmse:.2f}")
    print(f" -MAE  : {mae:.2f}")
    print(f" -MAPE : {mape:.2f}%")
    print(f" -R²   : {r2:.4f}")

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }

### **3.5 APPLYING CNN-LSTM**

In [ ]:
# APPLY CNN LSTM

# Number of time steps per sequence
sequence_length = 24

# Prepare CNN-LSTM sequences (features for each timestep)
X_cnn_lstm_dict, y_cnn_lstm_dict, feature_scaler_lstm, target_scaler_dict = cnnlstm_seqs(
    # Combines features (X) with targets (Y) into a DataFrames
    pd.concat([X, y], axis=1),
    targets,
    fcolumns,
    sequence_length
)

# Shape of sequence
print("Shape of a sequence with coordinates:", X_cnn_lstm_dict[targets[0]].shape)

# Dictionaries to store results for all targets
trained_models = {}
histories = {}
results = {}
test_sets = {}

# Total number of target variables
n_targets = len(targets)

# Train and evaluate model for each target
for idx, target_name in enumerate(targets, 1):
    print(f"\n--- Processing target {idx} out of {n_targets}: {target_name}")

    # Retrieve the prepared sequences for this target
    X_data = X_cnn_lstm_dict[target_name]
    y_data = y_cnn_lstm_dict[target_name]

    # Train the model with early stopping
    model, history, X_test, y_test = train_cnnlstm(
        target_name,
        X_data,
        y_data,
        callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
    )

    # Store the trained models and training history
    trained_models[target_name] = model
    histories[target_name] = history

    # Store the test set for this target
    test_sets[target_name] = (X_test, y_test)

    # Get the scaler for this target (for inverse transformation of predictions)
    target_scaler = target_scaler_dict[target_name]

    # Evaluate the performance of the model on test set
    metrics = evaluate_mtarget(
        model,
        X_test,
        y_test,
        target_scaler,
        target_name,
        idx,
        n_targets
    )

    # Save the metrics for the evaluated target
    results[target_name] = metrics

print("\nTraining and evaluation completed for all targets.")

Shape of a sequence with coordinates: (53400, 24, 24)

--- Processing target 1 out of 19: CYC_Clontarf James Larkin Rd

Training model for: CYC_Clontarf James Larkin Rd ===
Epoch 1/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 19s 157ms/step - loss: 0.0282 - mae: 0.0467 - val_loss: 0.0218 - val_mae: 0.0531
Epoch 2/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 119ms/step - loss: 0.0185 - mae: 0.0307 - val_loss: 0.0145 - val_mae: 0.0465
Epoch 3/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - loss: 0.0125 - mae: 0.0251 - val_loss: 0.0106 - val_mae: 0.0463
Epoch 4/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 120ms/step - loss: 0.0086 - mae: 0.0208 - val_loss: 0.0080 - val_mae: 0.0492
Epoch 5/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 119ms/step - loss: 0.0061 - mae: 0.0182 - val_loss: 0.0065 - val_mae: 0.0509
Epoch 6/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - loss: 0.0045 - mae: 0.0175 - val_loss: 0.0053 - val_mae: 0.0529
Epoch 7/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 120ms/step - loss: 0.0034 - mae: 0.0163 - val_loss: 0.0045 - val_mae: 

## **3.6 RESULTS**

In [ ]:
results = pd.DataFrame.from_dict(results, orient="index")
results.index.name = "Target"
results = results.reset_index()
print("\nSummary of results CNN-LSTM without coordinates:")
results


Summary of results CNN-LSTM without coordinates:


,Target,RMSE,MAE,MAPE,R2
0,CYC_Clontarf James Larkin Rd,18.883928,11.722347,62.211089,0.758905
1,CYC_Clontarf Pebble Beach Carpark,27.761920,17.818102,88.261566,0.743523
2,CYC_Grove Road Totem,41.386365,22.434457,47.366530,0.880121
3,CYC_North Strand Rd N B,0.000000,0.000000,NaN,1.000000
4,CYC_Richmond Street Cyclists 1 Merged,39.551493,24.278152,137.393276,0.265976
5,CYC_Richmond Street Cyclists 2 Merged,25.229261,14.982272,54.676945,0.648328
6,FTF_Aston Quay Merged,1694.930461,1517.101124,172.005987,-0.857670
7,FTF_Capel St Mary St Merged,1057.876128,916.473658,418.042156,-2.162089
8,FTF_College St Dame St Merged,278.115970,181.217228,61.174618,0.566426
9,FTF_Dame St Merged,457.553778,349.499001,390.951687,-1.051619


### **3.7 PREDICTIONS**

In [ ]:
# Dictionary with DataFrames for all targets
predictions = {}

for target_name in targets:
    print(f"\nPredicting values for: {target_name}")
    X_test, y_test = test_sets[target_name]
    target_scaler = target_scaler_dict[target_name]

    # Scale and descale predictions
    y_pred_scaled = trained_models[target_name].predict(X_test)
    y_pred = target_scaler.inverse_transform(y_pred_scaled)
    y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    # Predicted values flatten to 1D arrays
    y_pred = y_pred.reshape(-1)
    y_true = y_true.reshape(-1)

    # Index of start with date and time
    # Sequence length in LSTM sequences
    seq_len = sequence_length
    n_total = len(y_cnn_lstm_dict[target_name])
    n_train = int(n_total * 0.7)
    n_val = int(n_total * 0.15)
    start_test_idx = seq_len + n_train + n_val

    datest = data["date and time"].iloc[start_test_idx:].reset_index(drop=True)

    # DataFrame with only whole numbers and avoiding negative values
    predsDataFrame = pd.DataFrame({
        "date and time": datest,
        "Real": np.maximum(np.round(y_true).astype(int), 0),
        "CNNLSTM_nocoords": np.maximum(np.round(y_pred).astype(int), 0)
    })
    predictions[target_name] = predsDataFrame
    print(predsDataFrame.head())

# Merge all targets in a single DataFrame
allpreds = pd.concat(
    [dataframe.assign(Target=target) for target, dataframe in predictions.items()],
    ignore_index=True
)


Predicting values for: CYC_Clontarf James Larkin Rd
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step
         date and time  Real  CNNLSTM_nocoords
0  2024-03-07 06:00:00    20                23
1  2024-03-07 07:00:00    49                50
2  2024-03-07 08:00:00    70                67
3  2024-03-07 09:00:00    51                45
4  2024-03-07 10:00:00    31                41

Predicting values for: CYC_Clontarf Pebble Beach Carpark
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step
         date and time  Real  CNNLSTM_nocoords
0  2024-03-07 06:00:00    26                37
1  2024-03-07 07:00:00    75                67
2  2024-03-07 08:00:00   105                91
3  2024-03-07 09:00:00    75                71
4  2024-03-07 10:00:00    33                69

Predicting values for: CYC_Grove Road Totem
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step
         date and time  Real  CNNLSTM_nocoords
0  2024-03-07 06:00:00   125                77
1  2024-03-07 07:00:00   309               215
2  2024-03-07

In [ ]:
allpreds.head(20)

,date and time,Real,CNNLSTM_nocoords,Target
0,2024-03-07 06:00:00,20,23,CYC_Clontarf James Larkin Rd
1,2024-03-07 07:00:00,49,50,CYC_Clontarf James Larkin Rd
2,2024-03-07 08:00:00,70,67,CYC_Clontarf James Larkin Rd
3,2024-03-07 09:00:00,51,45,CYC_Clontarf James Larkin Rd
4,2024-03-07 10:00:00,31,41,CYC_Clontarf James Larkin Rd
5,2024-03-07 11:00:00,32,41,CYC_Clontarf James Larkin Rd
6,2024-03-07 12:00:00,36,44,CYC_Clontarf James Larkin Rd
7,2024-03-07 13:00:00,33,42,CYC_Clontarf James Larkin Rd
8,2024-03-07 14:00:00,23,43,CYC_Clontarf James Larkin Rd
9,2024-03-07 15:00:00,25,50,CYC_Clontarf James Larkin Rd


## **4. RESULTS TO AZURE BLOB STORAGE**

In [ ]:
def save_blob(data, filename):
    try:
        blob_name = f"{CONTAINER_NAME}/{filename}"
        csv_data = data.to_csv(index=False)
        with fs.open(blob_name, "w") as f:
            f.write(csv_data)
        print(f"Saved to {blob_name}")
    except Exception as e:
        print(f"Error saving data to blob storage: {str(e)}")
        return False

In [ ]:
CONTAINER_NAME = "results"

save_blob(results, "rcnnlstm_nocoords.csv")

Saved to results/rcnnlstm_nocoords.csv


In [ ]:
CONTAINER_NAME = "predictions"

save_blob(allpreds, "pred_cnnlstm_nocoords.csv")

Saved to predictions/pred_cnnlstm_nocoords.csv
